In [0]:
# Environment selection as dropdown
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="fq_dev_pnl",
    choices=["fq_dev_pnl", "fq_test_pnl", "fq_prod_pnl"],
    label="Select environment"
)

# Source selection as combobox
dbutils.widgets.combobox(
    name="source",
    defaultValue="NETSUITE",
    choices=["POSIST", "NETSUITE", "other"],
    label="Source"
)

# Domain selection as combobox
dbutils.widgets.combobox(
    name="domain",
    defaultValue="management_pnl",
    choices=["management_pnl"],
    label="Domain"
)

environment = dbutils.widgets.get("environment")
source = dbutils.widgets.get("source")
domain = dbutils.widgets.get("domain")

staging = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `fq_dev_extloc_staging`"
).select("url").collect()[0][0]

checkpoint = 'abfss://fq-dev-pnl-container@fqadfstoragedev.dfs.core.windows.net/checkpoints/'

In [0]:
%sql
select * from fq_dev_pnl_catalog.bronze.gl_report limit 1

In [0]:
%run "/Workspace/Users/tgh3@foodquest.ae/FoodQuest_PnL.git/FoodQuest Management P&L/Formula & Functions Management P&L"

In [0]:
df_brand_ho_rows = spark.read.table('fq_dev_pnl_catalog.bronze.brand_ho_allocation_cost')

df_exploded_keys = df_exploded.select(
    col("year"), 
    col("month"), 
    col("location").alias("netsuite_location_name")
).distinct()


df_brand_ho_rows_filtered = df_brand_ho_rows.join(
    df_exploded_keys,
    (df_brand_ho_rows["year"] == df_exploded_keys["year"]) &
    (df_brand_ho_rows["month"] == df_exploded_keys["month"]) &
    (df_brand_ho_rows["location"] == df_exploded_keys["netsuite_location_name"]),
    "inner"
).select(df_brand_ho_rows["*"])

# Perform union with filtered data
df_exploded = df_exploded.union(df_brand_ho_rows_filtered)

In [0]:
df = spark.read.option('multiline', False).format('json').load(f'{staging}/FoodQuest/Netsuite/GL_Report/ALBAIK/2026/JAN/gl_report.json')
exploded_df = (
            df.select(
                explode('results').alias('result')
            ).select('result.*')
        )
# display(exploded_df)


df_coa_master = spark.read.table("fq_dev_pnl_catalog.bronze.dim_coa_master")
df_location_master = spark.read.table("fq_dev_pnl_catalog.bronze.dim_location_master")

# Step 1: Join both master tables
df_all_masters = exploded_df.join(
    df_coa_master, 
    df_coa_master["account_number"].cast("string") == exploded_df["accountNo"], 
    'inner'
).join(
    df_location_master,
    col("location") == df_location_master.netsuite_location_name,
    'left'
)

df_final_netsuite = final_df(df_all_masters, 'actual')
df_final_netsuite.display()

In [0]:
%sql
CREATE TABLE IF NOT EXISTS fq_dev_pnl_catalog.silver.management_pnl (
  city STRING,
  management_sort_order INT,
  location_id INT,
  store_type STRING,
  major_group STRING,
  detail_total STRING COMMENT 'Detail/Total indicator',
  account_name STRING,
  zone STRING,
  sub_group STRING,
  management_details_total STRING,
  type STRING COMMENT 'Store/HO type',
  account_type STRING COMMENT 'Income/Expense type',
  store_open_date2 STRING,
  brand_id STRING,
  company_id STRING,
  parent_company STRING,
  country_code STRING,
  group_name STRING COMMENT 'Sales and Services Income group',
  management_group STRING,
  year INT,
  month STRING,
  netsuite_location_name STRING,
  mapped_name STRING,
  amount DECIMAL(18,2),
  budget_amount DECIMAL(18,2),
  py_amount DECIMAL(18,2) COMMENT 'Previous year amount'
)
USING DELTA
CLUSTER BY (year, month, netsuite_location_name, mapped_name)
LOCATION 'abfss://fq-dev-pnl-container@fqadfstoragedev.dfs.core.windows.net/silver/management_pnl' 
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
def merge_stream_management_pnl(df, i):
    try:
        exploded_df = (
            df.select(
                explode('results').alias('result')
            ).select('result.*')
        )
        
        management_pnl_upsert = enrich_json(exploded_df)
        management_pnl_upsert.createOrReplaceTempView("management_pnl_upsert_microbatch")
       
        df.sparkSession.sql("""
            MERGE INTO fq_dev_catalog.silver.management_pnl target
            USING (
                SELECT *
                FROM management_pnl_upsert_microbatch
            ) as source
            ON target.year = source.year
                AND target.month = source.month
                AND target.location = source.location
                AND target.account_name = source.account_name
                AND target.sum_order = source.sum_order
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        
        print(f"Successfully merged batch {i}")
        print(f"Batch {i}: {df.count()} rows")
        display(df.limit(10))  # Show first 10 rows

    # df.sparkSession.sql("""
    #     MERGE INTO fq_dev_catalog.silver.management_pnl target
    #     USING (
    #         SELECT *
    #         FROM (
    #             SELECT *, 
    #                 ROW_NUMBER() OVER (
    #                     PARTITION BY year, month, location, account_name, sum_order
    #                     ORDER BY year DESC  -- or add a load_time column
    #                 ) as rank
    #             FROM management_pnl_upsert_microbatch
    #         )
    #         WHERE rank = 1
    #     ) as source
    #     ON target.year = source.year
    #         AND target.month = source.month
    #         AND target.location = source.location
    #         AND target.account_name = source.account_name
    #         AND target.sum_order = source.sum_order
    #     WHEN MATCHED THEN UPDATE SET *
    #     WHEN NOT MATCHED THEN INSERT *
    # """)
    except Exception as e:
        print(f"Error in merge_stream: {e}")
        raise e

(spark.readStream
    # .option("schemaTrackingLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpointing_management_pnl/schema_management_pnl')
    .table("fq_dev_catalog.bronze.gl_report")
    .writeStream
    .foreachBatch(merge_stream_management_pnl)
    .option("mergeSchema", "true")
    .option('skipChangeCommits', "true")
    .option("checkpointLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpoint_silver_management_pnl3')  
    .trigger(availableNow=True)
    .start()
).awaitTermination()

time.sleep(20)

In [0]:
%sql
SELECT 
  COUNT(*) AS total_rows
FROM fq_dev_catalog.silver.management_pnl;

In [0]:
%sql
select * from fq_dev_catalog.silver.management_pnl limit 1

In [0]:
for query in spark.streams.active:
    query.stop()